In [19]:
import pickle


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, Flatten, Dense, Dropout, Input
from tensorflow.keras.regularizers import l1_l2
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import train_test_split
import os

In [ ]:
X_dcnn= np.load('/content/drive/MyDrive/NIDS/X_dcnn.npy')
y_labels =np.load('/content/drive/MyDrive/NIDS/y_labels.npy')

num_features = X_dcnn.shape[1]
print(f"Data loaded, input shape:{X_dcnn.shape}")

Data loaded, input shape:(1113112, 78, 1)


In [ ]:

X_train, X_test, y_train, y_test = train_test_split(
    X_dcnn, y_labels, test_size=0.2, random_state=42, stratify=y_labels
)
print(f"Training samples: {X_train.shape[0]}")
print(f"Testing samples: {X_test.shape[0]}")

Training samples: 890489
Testing samples: 222623


In [ ]:
print("Building the DCNN model...")
model = Sequential()
model.add(Input(shape=(num_features, 1)))
model.add(Conv1D(filters=128, kernel_size=3, activation='relu'))
model.add(Conv1D(filters=256, kernel_size=3, activation='relu'))
model.add(Flatten())

model.add(Dense(
    units=256,
    activation='relu',
    kernel_regularizer=l1_l2(l1=0.01, l2=0.01)
))
model.add(Dropout(0.1))

model.add(Dense(units=2, activation='softmax'))

Building the DCNN model...


In [ ]:
optimizer = Adam(learning_rate=0.0001)
model.compile(
    optimizer=optimizer,
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d_2 (Conv1D)               │ (None, 76, 128)        │           512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_3 (Conv1D)               │ (None, 74, 256)        │        98,560 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 18944)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 256)            │     4,849,920 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 2)              │           514 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,949,506 (18.88 MB)

 Trainable params: 4,949,506 (18.88 MB)

 Non-trainable params: 0 (0.00 B)

In [14]:
print("Starting training...")
history = model.fit(
    X_train, y_train,
    epochs=30,
    batch_size=256,
    validation_data=(X_test, y_test),
    verbose=1
)

Starting training...
Epoch 1/30
3479/3479 ━━━━━━━━━━━━━━━━━━━━ 39s 10ms/step - accuracy: 0.8574 - loss: 8.3893 - val_accuracy: 0.9218 - val_loss: 0.7854
Epoch 2/30
3479/3479 ━━━━━━━━━━━━━━━━━━━━ 35s 10ms/step - accuracy: 0.9371 - loss: 0.7456 - val_accuracy: 0.9460 - val_loss: 0.7020
Epoch 3/30
3479/3479 ━━━━━━━━━━━━━━━━━━━━ 34s 10ms/step - accuracy: 0.9497 - loss: 0.6910 - val_accuracy: 0.9511 - val_loss: 0.6669
Epoch 4/30
3479/3479 ━━━━━━━━━━━━━━━━━━━━ 34s 10ms/step - accuracy: 0.9593 - loss: 0.6651 - val_accuracy: 0.9631 - val_loss: 0.6511
Epoch 5/30
3479/3479 ━━━━━━━━━━━━━━━━━━━━ 34s 10ms/step - accuracy: 0.9645 - loss: 0.6518 - val_accuracy: 0.9686 - val_loss: 0.6343
Epoch 6/30
3479/3479 ━━━━━━━━━━━━━━━━━━━━ 34s 10ms/step - accuracy: 0.9687 - loss: 0.6456 - val_accuracy: 0.9755 - val_loss: 0.6317
Epoch 7/30
3479/3479 ━━━━━━━━━━━━━━━━━━━━ 34s 10ms/step - accuracy: 0.9711 - loss: 0.6473 - val_accuracy: 0.9797 - val_loss: 0.6375
Epoch 8/30
3479/3479 ━━━━━━━━━━━━━━━━━━━━ 34s 10ms/step

In [15]:
print("\nEvaluating model on test data...")
test_loss, test_accuracy = model.evaluate(X_test, y_test, verbose=1)

print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")


Evaluating model on test data...
6957/6957 ━━━━━━━━━━━━━━━━━━━━ 18s 3ms/step - accuracy: 0.9858 - loss: 0.6118
Test Loss: 0.6118
Test Accuracy: 0.9858


In [16]:
from sklearn.metrics import classification_report, confusion_matrix
y_pred_probs = model.predict(X_test)
y_pred = np.argmax(y_pred_probs, axis=1)

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

6957/6957 ━━━━━━━━━━━━━━━━━━━━ 12s 2ms/step

Classification Report:
              precision    recall  f1-score   support

           0       0.99      0.98      0.99    111312
           1       0.98      0.99      0.99    111311

    accuracy                           0.99    222623
   macro avg       0.99      0.99      0.99    222623
weighted avg       0.99      0.99      0.99    222623


Confusion Matrix:
[[109402   1910]
 [  1241 110070]]


In [17]:
model.save('/content/drive/MyDrive/dcnn_nids_model.h5')
print("Model saved successfully!")

Model saved successfully!


In [20]:
with open('/content/drive/MyDrive/training_history.pkl', 'wb') as f:
    pickle.dump(history.history, f)

print("Training history saved!")

Training history saved!
